# 🎮 Colab Gaming PC (L4 GPU / High-Memory) - クラウドゲーミング環境

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/9z2fqjckjv-bot/ColabTool/blob/main/GamingPC/GamingPC_Colab.ipynb)

Google Colab の強力な **NVIDIA L4 GPU (24GB VRAM) & ハイメモリ (53GB RAM)** を活用し、本格的なクラウドゲーミングワークステーションをワンクリックで構築・稼働させる自動化ノートブックです。

---

### 🌟 本環境の特徴
- **🇯🇵 日本語 UI & 日本標準時 (JST / 東京)**: Xfce4 デスクトップ全体・時計・Google Noto CJK / カラー絵文字フォントを完全日本語化。
- **💻 MacBook 最適化解像度 (16:10 / 1680x1050)**: MacBook の画面比率に合わせてジャストフィット。
- **⚡ Web Audio API 超低遅延・リアルタイム音同期**: ブラウザ遅延を完全排除。PCM 16bit 44.1kHz で映像と音が 0.1秒未満で同期（YouTube・ゲームのリップシンク対応）。
- **🛡️ Steam & 一般ユーザー権限の完全整備**: root制限を回避する一般ユーザー (`colabuser`) とセーフティラッパーを自動構成。
- **🌐 Google Chrome マルチプロファイル & 充実のランチャー**:
  - **Chrome Profile 1 / 2 / 3**: 個別の独立プロファイルでタブやアカウントを使い分け。
  - **Steam**: Hoi4、ChillwithYou などをプレイ。
  - **Minecraft (Prism Launcher)**: Modpack や各バージョンをワンクリック管理。
  - **Xbox Cloud Gaming / Roblox / Discord / OBS Studio**
- **💾 Google Drive データ永続化**: ワールド、セーブデータ、録画、プロファイルを自動バックアップ。

---
## 📍 ステップ 0: リージョン確認＆東京リージョン（Tokyo, JP）判定

接続先データセンターの所在地（リージョン）とネットワーク情報をチェックします。

In [ ]:
#@title 📍 ステップ 0: リージョン確認＆東京リージョン判定
import urllib.request
import json
from IPython.display import display, HTML

print("🔍 接続先データセンターのリージョン情報を照会中...")
region_info = {}
try:
    req = urllib.request.Request("https://ipinfo.io/json", headers={"User-Agent": "curl/7.68.0"})
    with urllib.request.urlopen(req, timeout=5) as response:
        region_info = json.loads(response.read().decode())
except Exception as e:
    region_info = {"city": "不明", "country": "不明", "org": str(e)}

city = region_info.get("city", "不明")
country = region_info.get("country", "不明")
ip = region_info.get("ip", "-")
org = region_info.get("org", "Google LLC")

is_tokyo = (city.lower() == "tokyo" or country == "JP")
status_color = "#10b981" if is_tokyo else "#f59e0b"
status_title = "🎉 東京リージョン（Tokyo, JP）で接続中！" if is_tokyo else f"⚠️ 現在は {city}, {country} リージョンで接続されています"
status_msg = "遅延（Ping 5〜15ms）が最小のベストな環境です。そのままステップ 1 へ進んでください。" if is_tokyo else (
    "シンガポール等のアジア近郊リージョン（約60ms）でも利用可能ですが、より低遅延な東京リージョンを狙うには上部メニューの「ランタイム」➔「ランタイムを接続解除して削除」を行い、再接続をお試しください。"
)

display(HTML(f'''
<div style="background: #0f172a; padding: 22px; border-radius: 12px; border: 1px solid #334155; border-left: 6px solid {status_color}; font-family: sans-serif;">
  <h3 style="color: {status_color}; margin: 0 0 10px 0; font-size: 18px;">{status_title}</h3>
  <p style="color: #e2e8f0; font-size: 14px; margin-bottom: 12px;">{status_msg}</p>
  <div style="background: #1e293b; padding: 12px; border-radius: 8px; font-size: 13px; color: #94a3b8;">
    <strong>IP:</strong> {ip} | <strong>所在地:</strong> {city}, {country} | <strong>ネットワーク:</strong> {org}
  </div>
</div>
'''))


---
## 🚀 ステップ 1: 【ワンクリック起動】Gaming PC

下のセルを実行するだけで、**日本語環境・MacBook 16:10 解像度・超低遅延音同期・全ゲームランチャー** が全自動で立ち上がります。  
約1〜2分でセルの実行は正常完了します。

> [!TIP]
> 実行完了後、セル下部に表示される **「🚀 今すぐデスクトップを開く」** ボタンをクリックしてください！

In [ ]:
#@title 🎮 ステップ 1: 【ワンクリック起動】Gaming PC
import os
import subprocess
import time
from IPython.display import display, HTML
from google.colab.output import eval_js

#@markdown ### 💻 画面解像度の選択 (MacBook 向け 16:10 推奨)
SCREEN_RES = "1680x1050" #@param ["1680x1050", "1920x1200", "1920x1080", "1440x900"]

#@markdown ### 💾 Google Drive バックアップ
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}

# 1. Google Drive の自動マウント
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("📂 Google Drive をマウント中...")
            drive.mount("/content/drive")
            print("✅ Google Drive マウント完了")
        else:
            print("📂 Google Drive は既にマウントされています。")
    except Exception as e:
        print("⚠️ Google Driveマウントをスキップ:", e)

# 2. GitHub から最新スクリプトを確実に同期
REPO_DIR = "/content/ColabTool"
print("📦 最新環境構成を取得中...")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/9z2fqjckjv-bot/ColabTool.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main", "-q"], check=False)

# 3. 堅牢起動スクリプトの実行（リアルタイムログ表示）
launch_script = os.path.join(REPO_DIR, "GamingPC/scripts/launch.sh")
proc = subprocess.Popen(["bash", launch_script, SCREEN_RES], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"起動エラーが発生しました (exit code: {proc.returncode})")

# 4. Colab 公式プロキシ URL を生成して表示
colab_base = eval_js("google.colab.kernel.proxyPort(6080)").rstrip("/")
vnc_url = f"{colab_base}/vnc.html?autoconnect=true&resize=scale"

display(HTML(f'''
<div style="background: #0f172a; padding: 24px; border-radius: 12px; border: 1px solid #334155; border-left: 6px solid #10b981; font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.4); margin-top: 15px;">
  <h3 style="color: #10b981; margin: 0 0 10px 0; font-size: 20px; display: flex; align-items: center; gap: 8px;">
    🎉 Gaming PC 起動完了！（日本語・音同期・MacBook 16:10 最適化）
  </h3>
  <p style="color: #cbd5e1; font-size: 14px; margin-bottom: 18px;">
    下のボタンをクリックすると、ブラウザの新しいタブで <strong>日本語デスクトップ</strong> が開きます：
  </p>
  <a href="{vnc_url}" target="_blank" style="display: inline-block; background: #10b981; color: #042f2e; padding: 14px 28px; border-radius: 8px; font-size: 16px; font-weight: bold; text-decoration: none; box-shadow: 0 4px 12px rgba(16, 185, 129, 0.3);">
    🚀 今すぐデスクトップを開く
  </a>
  <p style="font-size: 12px; color: #94a3b8; margin-top: 14px;">
    ※ デスクトップ画面内を一度クリックすると、右上に <strong>[LIVE SOUND]</strong> が表示され、YouTube やゲームの音が Mac からリアルタイムに再生されます。
  </p>
</div>
'''))


---
## 💾 ステップ 2: Google Drive 即時同期 (手動バックアップ)

Steam セーブ、Chrome プロファイル、各種ゲーム設定を Google Drive へバックアップします。

In [ ]:
#@title 💾 ステップ 2: Google Drive 即時同期
import subprocess
import sys

script_path = "/content/ColabTool/GamingPC/scripts/backup_restore.py"
if os.path.exists(script_path):
    subprocess.run([sys.executable, script_path, "backup"])
    subprocess.run([sys.executable, script_path, "status"])
else:
    print("同期スクリプトが見つかりません。先にステップ 1 を実行してください。")


---
## 🛑 ステップ 3: 安全なシャットダウン＆データ保存

作業終了時にこのセルを実行してください。セーブデータを Google Drive に保存して全プロセスを安全に停止します。

In [ ]:
#@title 🛑 ステップ 3: 安全なシャットダウン＆データ保存
import subprocess
import os

script_path = "/content/ColabTool/GamingPC/scripts/shutdown.sh"
if os.path.exists(script_path):
    subprocess.run(["bash", script_path])
else:
    print("終了スクリプトが見つかりません。")
